In [3]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [4]:
import torch
import torch.nn as nn

from torchvision import models

from data.dataloader import get_dataloaders

In [8]:
print(models)

<module 'torchvision.models' from 'c:\\Users\\TANISHA\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\torchvision\\models\\__init__.py'>


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [11]:
train_loader, val_loader, test_loader, class_names = get_dataloaders()

In [12]:
weights = models.ResNet50_Weights.DEFAULT

model = models.resnet50(weights=weights)

In [13]:
num_classes = len(class_names)

model.fc = torch.nn.Linear(
    model.fc.in_features,
    num_classes
)

model = model.to(device)

print(model.fc)

Linear(in_features=2048, out_features=30, bias=True)


In [14]:
criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

print("Loss:", criterion)
print("Optimizer:", optimizer)

Loss: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)


In [15]:
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = model(images)

print("Output shape:", outputs.shape)

Output shape: torch.Size([32, 30])


In [16]:
epochs = 10

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

best_val_acc = 0

In [17]:
for epoch in range(epochs):

    # ---------------- TRAIN ----------------
    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # ---------------- VALIDATION ----------------
    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = running_loss / len(val_loader)
    val_acc = 100 * correct / total

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(model.state_dict(), "../models/resnet50_best.pth")

Epoch 1/10 | Train Loss: 1.1481 | Train Acc: 69.52% | Val Loss: 0.5308 | Val Acc: 82.09%
Epoch 2/10 | Train Loss: 0.3386 | Train Acc: 88.75% | Val Loss: 0.4499 | Val Acc: 84.53%
Epoch 3/10 | Train Loss: 0.1834 | Train Acc: 93.38% | Val Loss: 0.4584 | Val Acc: 84.31%
Epoch 4/10 | Train Loss: 0.1278 | Train Acc: 95.58% | Val Loss: 0.4610 | Val Acc: 85.96%
Epoch 5/10 | Train Loss: 0.1136 | Train Acc: 95.84% | Val Loss: 0.4846 | Val Acc: 85.96%
Epoch 6/10 | Train Loss: 0.1101 | Train Acc: 95.74% | Val Loss: 0.5484 | Val Acc: 84.62%
Epoch 7/10 | Train Loss: 0.1030 | Train Acc: 95.95% | Val Loss: 0.4951 | Val Acc: 86.18%
Epoch 8/10 | Train Loss: 0.0882 | Train Acc: 96.53% | Val Loss: 0.5552 | Val Acc: 85.42%
Epoch 9/10 | Train Loss: 0.0822 | Train Acc: 96.63% | Val Loss: 0.5717 | Val Acc: 84.36%
Epoch 10/10 | Train Loss: 0.0857 | Train Acc: 96.61% | Val Loss: 0.5860 | Val Acc: 85.64%
